# NimbusPay ticket triage: fine-tuning Qwen2.5-1.5B-Instruct with LoRA / QLoRA

**Task.** Read a support ticket, return an 8-key JSON triage record. The model gets only a fixed 1-line system prompt + the ticket at test time, so it must *learn* the business rules (priority, null rules, paise conversion, date resolution, id normalisation) from data.

**Setup.** Free Colab **T4** (16 GB, fp16 only, no bf16). Base model `Qwen/Qwen2.5-1.5B-Instruct`. Own training loop (PyTorch + PEFT); no Trainer/TRL.

| Part | What |
|---|---|
| A | Audit and clean `train.jsonl` (rule checker derived from SCHEMA.md) |
| B | Baselines on dev: fixed prompt, and rules-in-prompt |
| C | Pipeline evidence: chat template, loss masking, sequence length, parameter budget |
| D | Controlled experiments (one change at a time) |
| E | Error analysis, final model, test predictions |

**To run:** upload the `candidate_pack` folder to Google Drive at `MyDrive/nimbus/candidate_pack`, select *Runtime > Change runtime type > T4 GPU*, then *Run all*. Adapters, predictions and the results log are saved to `MyDrive/nimbus/work`.

**AI assistance.** An AI coding assistant (an LLM agent) was used to write the audit scripts, this notebook and the training module; every number below is produced by the cells here and was reviewed by me.

In [ ]:
# 0. Environment ---------------------------------------------------------------------------------------------------
%pip install -q -U transformers peft accelerate bitsandbytes
# Colab ships torchao 0.10.0 but the current peft refuses to import with torchao < 0.16 (ImportError in get_peft_model).
# We do not use torchao, so remove it.
%pip uninstall -y -q torchao

In [ ]:
import os, sys, json, time, gc, random, csv, collections, subprocess
import numpy as np, pandas as pd, torch, transformers, peft
import bitsandbytes as bnb
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| bitsandbytes", bnb.__version__)
print("GPU:", torch.cuda.get_device_name(0), "| memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if "T4" not in torch.cuda.get_device_name(0):
    print("WARNING: the assignment targets a Colab T4; timings and memory below are for", torch.cuda.get_device_name(0))

from google.colab import drive
drive.mount("/content/drive")
PACK = "/content/drive/MyDrive/nimbus/candidate_pack"      # contains data/ and score.py
WORK = "/content/drive/MyDrive/nimbus/work"                # adapters, predictions, results log
for d in ("adapters", "preds", "logs"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
os.makedirs("/content/nimbus", exist_ok=True)
assert os.path.exists(f"{PACK}/data/train.jsonl") and os.path.exists(f"{PACK}/score.py"), "upload candidate_pack to MyDrive/nimbus/candidate_pack"

torch 2.14.1+cu130 | transformers 5.22.2 | peft 0.21.2 | bitsandbytes 0.50.2
GPU: Tesla T4 | memory GB: 14.6
Mounted at /content/drive


The three modules below are written to disk and imported, so the notebook is self-contained.
`nimbus_lib` = SCHEMA.md rules as code (used to audit and clean); `nimbus_clean` = the cleaning decisions; `nimbus_train` = data -> tensors, loss, training loop, greedy generation.

In [ ]:
%%writefile /content/nimbus/nimbus_lib.py
"""Shared helpers for the audit: parse a ticket and derive each label field from SCHEMA.md rules.

Everything here uses only SCHEMA.md and the ticket text. No dev/test labels are used.
"""
import json, re, datetime as dt
from decimal import Decimal

ROOT = "E:/Fine_Tuneing-Assigh/training_assessment/candidate_pack/data/"
KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
NULL_CATS = {"account_access", "kyc", "other"}  # transaction fields must be null for these


def load(name):
    import os
    path = name if os.path.isabs(name) else ROOT + name
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


# ---------------------------------------------------------------- ticket parsing
HDR = re.compile(r"^Received: (\d{4})-(\d{2})-(\d{2}) \((\w{3})\)\nSource: ([\w-]+)\n\n", re.S)


def parse_ticket(user):
    m = HDR.match(user)
    received = dt.date(int(m.group(1)), int(m.group(2)), int(m.group(3)))
    body = user[m.end():]
    own = "\n".join(l for l in body.split("\n") if not l.lstrip().startswith(">"))  # drop quoted earlier emails
    return received, m.group(5), body, own


# ---------------------------------------------------------------- amounts
NUM = r"\d{1,3}(?:,\d{2,3})+(?:\.\d+)?|\d+(?:\.\d+)?"
UNIT = r"k|lakhs?|lacs?"
AMT = re.compile(
    rf"(?P<pre>\u20b9|\bRs\.?|\bINR)\s*(?P<n1>{NUM})(?:\s*(?P<u1>{UNIT})\b)?"
    rf"|(?<![A-Za-z0-9.,])(?P<n2>{NUM})\s*(?P<u2>{UNIT})\b"
    rf"|(?<![A-Za-z0-9.,])(?P<n3>{NUM})\s*(?:rupees|rupaye|rupaiye)\b",
    re.I,
)
DISTRACT_LEFT = re.compile(r"balance|daily limit|limit|\bfees?\b|charges?\b|\bmein abhi\b|\babhi\b", re.I)
DISTRACT_RIGHT = re.compile(r"^\s*(?:right now|pade hain|baki|bacha)", re.I)


def to_paise(num, unit):
    v = Decimal(num.replace(",", ""))
    u = (unit or "").lower()
    if u == "k":
        v *= 1000
    elif u.startswith("lakh") or u.startswith("lac"):
        v *= 100000
    return int(v * 100)


def amounts(text):
    """All currency-like amounts in text -> list of dict(paise, raw, distractor)."""
    out = []
    for m in AMT.finditer(text):
        num = m.group("n1") or m.group("n2") or m.group("n3")
        unit = m.group("u1") or m.group("u2")
        left = text[: m.start()]
        cut = max(left.rfind(". "), left.rfind("! "), left.rfind("? "), left.rfind("\n"))
        sent_left = left[cut + 1:]
        dis = bool(DISTRACT_LEFT.search(sent_left)) or bool(DISTRACT_RIGHT.match(text[m.end():]))
        out.append({"paise": to_paise(num, unit), "raw": m.group(0), "distractor": dis})
    return out


# ---------------------------------------------------------------- dates
MON = {m: i for i, m in enumerate(["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"], 1)}
MONRE = r"(Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|June?|July?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?)"
ABS_NUM = re.compile(r"(?<![\d/-])(\d{1,2})[/-](\d{1,2})[/-](\d{4}|\d{2})(?![\d/-])")
ABS_DM = re.compile(rf"(?<!\d)(\d{{1,2}})(?:st|nd|rd|th)?\s+{MONRE}\b(?:,?\s+(\d{{4}}))?", re.I)
ABS_MD = re.compile(
    rf"\b{MONRE}\.?\s+(\d{{1,2}})(?:st|nd|rd|th)?\b(?!\s*:)(?!\.\d)(?!\s*(?:k\b|lakh|lac\b|rupe))(?:,?\s+(\d{{4}}))?", re.I
)  # "Aug 14" / "August 14th 2026"; not "Aug. 1.3k" (a sentence end followed by an amount)
REL = [
    (re.compile(r"day before yesterday|\bparso\b", re.I), -2),
    (re.compile(r"(?<!day before )\byesterday\b|\bkal\b", re.I), -1),
    (re.compile(r"\btoday\b|\baaj\b", re.I), 0),
]
REL_N = re.compile(r"\b(\d+)\s+(?:days?\s+ago|din\s+pehle)", re.I)


def _safe(y, m, d):
    try:
        return dt.date(y, m, d)
    except ValueError:
        return None


def _no_year(received, m, d):
    c = _safe(received.year, m, d)
    if c is None:
        return None
    return c if c <= received else _safe(received.year - 1, m, d)


def dates(text, received):
    """Candidate transaction dates mentioned in text -> list of (date, kind, raw)."""
    out = []
    for m in ABS_NUM.finditer(text):
        d, mo, y = int(m.group(1)), int(m.group(2)), m.group(3)
        y = int(y) + 2000 if len(y) == 2 else int(y)
        dd = _safe(y, mo, d)
        if dd:
            out.append((dd, "abs_num", m.group(0)))
    for m in ABS_DM.finditer(text):
        d, mo, y = int(m.group(1)), MON[m.group(2)[:3].lower()], m.group(3)
        dd = _safe(int(y), mo, d) if y else _no_year(received, mo, d)
        if dd:
            out.append((dd, "abs_dm", m.group(0)))
    for m in ABS_MD.finditer(text):
        mo, d, y = MON[m.group(1)[:3].lower()], int(m.group(2)), m.group(3)
        dd = _safe(int(y), mo, d) if y else _no_year(received, mo, d)
        if dd:
            out.append((dd, "abs_md", m.group(0)))
    for rx, off in REL:
        for m in rx.finditer(text):
            out.append((received + dt.timedelta(days=off), "rel", m.group(0)))
    for m in REL_N.finditer(text):
        out.append((received - dt.timedelta(days=int(m.group(1))), "rel", m.group(0)))
    return out


# ---------------------------------------------------------------- txn id
ID_RX = re.compile(r"(?<![A-Za-z])N\s*P(?:[\s-]*\d){10}(?!\d)", re.I)


def txn_ids(text):
    return ["NP" + re.sub(r"\D", "", m.group(0)) for m in ID_RX.finditer(text)]


# ---------------------------------------------------------------- channel
CH_STRICT = [
    (re.compile(r"Mode:\s*(upi|net ?banking|internet banking|wallet|credit card|debit card|card)\b", re.I), 1),
    (re.compile(r"It was an? (upi|net ?banking|internet banking|wallet|credit card|debit card|card) transaction", re.I), 1),
    (re.compile(r"\b(upi|net ?banking|wallet|card|credit card|debit card) se hua tha", re.I), 1),
    (re.compile(r"\b(credit card|debit card|card) wala transaction tha", re.I), 1),
    (re.compile(r"(from the NimbusPay wallet)", re.I), 1),
]
CH_LOOSE = [
    ("upi", re.compile(r"\bUPI\b", re.I)),
    ("netbanking", re.compile(r"net\s?banking|internet banking", re.I)),
    ("wallet", re.compile(r"\bwallet\b", re.I)),
    ("card", re.compile(r"credit card|debit card|card payment|card transaction|\bvisa\b|mastercard|\bcard\b", re.I)),
]


def _canon_ch(s):
    s = s.lower()
    if "upi" in s:
        return "upi"
    if "bank" in s:
        return "netbanking"
    if "wallet" in s:
        return "wallet"
    return "card"


def channels_strict(text):
    return sorted({_canon_ch(m.group(1)) for rx, _ in CH_STRICT for m in rx.finditer(text)})


def channels_loose(text):
    return sorted({n for n, rx in CH_LOOSE if rx.search(text)})


# ---------------------------------------------------------------- language, contact, legal
HI_WORDS = set("""hai hain hoon tha thi the kya kaise kyun kyu nahi nahin mera meri mere mujhe aap apna apni ko ka ki ke se par lekin
paise paisa wapas kat kaat gaye gaya hua hui ho kar karo karna karun kiya raha rahi rahe abhi tak jaldi dhanyavaad namaste bhai ji baar
teesri pehle pichla din rakam kitna kinta chahiye jaunga dunga bol bata batao dikha dikh mein kuch koi aur bhi hi toh to ek do
saamaan ghar phir fir chuka chala chali samajh hua hoga kijiye kijie kare kar sakta sakte""".split())
HI_STRONG = set("""hai hain hoon kya kaise nahi nahin mera meri mere mujhe lekin paise paisa wapas kat kaat gaye gaya raha rahi abhi jaldi
dhanyavaad namaste baar teesri pehle rakam kitna chahiye jaunga dunga bol bata karo karna karun kiya hua tha hoon""".split())


def hindi_score(text):
    toks = re.findall(r"[a-z]+", text.lower())
    return sum(t in HI_STRONG for t in toks), len(toks)


CONTACT = re.compile(
    r"third time|teesri baar|already written|contacted support|second complaint|again and again|baar baar|pehle bhi complaint",
    re.I,
)
TICKET_REF = re.compile(r"earlier ticket number|pichla ticket number", re.I)
LEGAL = re.compile(r"\bRBI\b|ombudsman|consumer court|lawyer|legal notice|legal action", re.I)


# ---------------------------------------------------------------- rules from SCHEMA.md
def exp_priority(cat, amount, legal):
    if cat == "fraud" or (amount is not None and amount >= 5_000_000) or legal:
        return "P1"
    if (cat in ("payment_failed", "refund") and amount is not None and amount >= 500_000) or cat == "account_access":
        return "P2"
    return "P3"


def exp_needs_human(priority, contacted):
    return priority == "P1" or contacted


def parse_label(content):
    try:
        lab = json.loads(content)
    except Exception:
        return None
    return lab if isinstance(lab, dict) else None


# ---------------------------------------------------------------- full derivation (spec + text -> expected label fields)
SCRATCH = re.compile(r"scratch\s*card|srcatch\s*card", re.I)


def channel_from_text(own):
    st = channels_strict(own)
    if len(st) == 1:
        return st[0]
    if len(st) > 1:
        return None
    lo = channels_loose(SCRATCH.sub(" ", own))
    return lo[0] if len(lo) == 1 else None


def derive(user, category):
    """Expected values for every deterministic field, given the (trusted) category.

    Returns dict with the fields plus 'ambiguous' = list of fields where the text gave >1 candidate.
    """
    rec, src, body, own = parse_ticket(user)
    amb = []
    if category in NULL_CATS:
        amount = date = tid = channel = None
    else:
        main = sorted({a["paise"] for a in amounts(own) if not a["distractor"]})
        if len(main) > 1:
            amb.append("amount_paise")
        amount = main[0] if len(main) == 1 else None
        cands = sorted({d.isoformat() for d, _, _ in dates(own, rec)})
        if len(cands) > 1:
            amb.append("txn_date")
        date = cands[0] if len(cands) == 1 else None
        ids = sorted(set(txn_ids(own)))
        if len(ids) > 1:
            amb.append("txn_id")
        tid = ids[0] if len(ids) == 1 else None
        channel = channel_from_text(own)
    legal, contacted = bool(LEGAL.search(own)), bool(CONTACT.search(own))
    prio = exp_priority(category, amount, legal)
    hs, _ = hindi_score(own)
    return {
        "category": category, "priority": prio, "amount_paise": amount, "txn_date": date, "txn_id": tid,
        "channel": channel, "language": "hi-en" if hs >= 2 else "en",
        "needs_human": exp_needs_human(prio, contacted), "ambiguous": amb,
        "_legal": legal, "_contacted": contacted,
    }

Writing /content/nimbus/nimbus_lib.py


In [ ]:
%%writefile /content/nimbus/nimbus_clean.py
"""Cleaning of train.jsonl (Part A). Uses only train.jsonl + SCHEMA.md rules (nimbus_lib.derive).

One exception, disclosed in the report: dev *inputs* (never dev labels) are used to find train rows that are copies of
dev tickets, because training on them would break the "never train on dev" rule.
"""
import json, re, collections
from nimbus_lib import *

JUNK = re.compile(r"^\s*(?:test|asdf asdf|test test \d+)?\s*$", re.I)
LEGACY_PRIO = {"low": "P3", "medium": "P2", "high": "P1"}


def norm_id(s):
    return "NP" + re.sub(r"\D", "", s) if s else s


def clean_row(r):
    """Return (new_label_dict, [fix reasons]) or (None, [drop reasons])."""
    user = r["messages"][1]["content"]
    lab = parse_label(r["messages"][2]["content"])
    fixes = []

    # ---- v1 (legacy) format: prio low/medium/high + dd/mm/yyyy dates
    if "prio" in lab:
        old = lab["prio"]
        lab = {("priority" if k == "prio" else k): v for k, v in lab.items()}
        lab["priority"] = LEGACY_PRIO[old]
        if lab["txn_date"]:
            dd, mm, yy = lab["txn_date"].split("/")
            lab["txn_date"] = f"{yy}-{mm}-{dd}"
        fixes.append(f"legacy v1 format: 'prio':'{old}'->'priority':'{lab['priority']}', date dd/mm/yyyy->ISO")
    lab = {k: lab[k] for k in KEYS}   # canonical key order
    d = derive(user, lab["category"])
    if d["ambiguous"]:
        return None, [f"ambiguous text for {d['ambiguous']}; label cannot be verified"]

    # ---- amount: rupees stored instead of paise
    if lab["amount_paise"] != d["amount_paise"]:
        t = d["amount_paise"]
        if lab["amount_paise"] is not None and t is not None and lab["amount_paise"] == t // 100:
            fixes.append(f"amount_paise stored in rupees ({lab['amount_paise']}) instead of paise ({t})")
            lab["amount_paise"] = t
        else:
            return None, [f"amount_paise {lab['amount_paise']} unexplained (text gives {t})"]
    # ---- txn_id: raw text copied instead of normalised NP+10 digits
    if lab["txn_id"] != d["txn_id"]:
        if lab["txn_id"] is not None and norm_id(lab["txn_id"]) == d["txn_id"]:
            fixes.append(f"txn_id not normalised ('{lab['txn_id']}' -> '{d['txn_id']}')")
            lab["txn_id"] = d["txn_id"]
        else:
            return None, [f"txn_id {lab['txn_id']} unexplained (text gives {d['txn_id']})"]
    # ---- remaining deterministic fields must already agree
    for k in ("txn_date", "channel", "language"):
        if lab[k] != d[k]:
            return None, [f"{k} {lab[k]!r} unexplained (text/spec gives {d[k]!r})"]
    # ---- priority: recompute from spec. Evidence that the *priority* column is the corrupted one:
    #      needs_human always agrees with the spec-derived priority, never with the bad label.
    if lab["needs_human"] != d["needs_human"]:
        return None, [f"needs_human {lab['needs_human']} unexplained (spec gives {d['needs_human']})"]
    if lab["priority"] != d["priority"]:
        fixes.append(
            f"priority {lab['priority']} contradicts spec -> {d['priority']} "
            f"(category={lab['category']}, amount_paise={lab['amount_paise']}, legal_threat={d['_legal']}); needs_human agrees with {d['priority']}"
        )
        lab["priority"] = d["priority"]
    return lab, fixes


def clean_dataset(rows, dev_rows):
    """-> (kept_rows, log[(id, action, reason)], stats Counter)"""
    dev_inputs = {r["messages"][1]["content"]: r["id"] for r in dev_rows}
    log, kept, seen_full = [], [], {}
    stats = collections.Counter()
    for r in rows:
        rid, user = r["id"], r["messages"][1]["content"]
        own = parse_ticket(user)[3] if HDR.match(user) else ""
        if user in dev_inputs:                                   # 1. leakage of dev tickets into train
            log.append((rid, "drop", f"leakage: ticket text identical to dev ticket {dev_inputs[user]} (never train on dev)"))
            stats["drop_dev_leak"] += 1
            continue
        if JUNK.match(own):                                      # 2. junk / empty ticket with an unsupported label
            log.append((rid, "drop", f"junk ticket (body={own.strip()[:15]!r}): label has fields the text cannot support"))
            stats["drop_junk"] += 1
            continue
        if parse_label(r["messages"][2]["content"]) is None:     # 3. truncated label
            log.append((rid, "drop", "assistant label is truncated / invalid JSON"))
            stats["drop_truncated"] += 1
            continue
        if user in seen_full:                                    # 4. exact duplicate of an earlier kept ticket
            log.append((rid, "drop", f"exact duplicate of {seen_full[user]} (same text and label)"))
            stats["drop_duplicate"] += 1
            continue
        lab, notes = clean_row(r)
        if lab is None:
            log.append((rid, "drop", "; ".join(notes)))
            stats["drop_unexplained"] += 1
            continue
        if notes:
            log.append((rid, "fix", "; ".join(notes)))
            for key, pat in (("fix_legacy", "legacy v1"), ("fix_amount", "stored in rupees"), ("fix_txn_id", "not normalised"), ("fix_priority", "contradicts spec")):
                if pat in notes[0]:
                    stats[key] += 1
        seen_full[user] = rid
        kept.append({"id": rid, "messages": [r["messages"][0], r["messages"][1], {"role": "assistant", "content": json.dumps(lab)}]})
    return kept, log, stats

Writing /content/nimbus/nimbus_clean.py


In [ ]:
%%writefile /content/nimbus/nimbus_train.py
"""Training / generation helpers for the NimbusPay triage fine-tune (Qwen2.5-1.5B-Instruct + LoRA / QLoRA).

Why a hand-written loop instead of Trainer/TRL
  * API-stable across library versions (Colab upgrades them under us),
  * the loss is computed ONLY on the assistant tokens and ONLY at those positions, so the 152k-vocab logits are never
    materialised for prompt tokens (a 2,200-token quoted-thread ticket would otherwise need ~1.3 GB of fp32 logits),
  * every line is explainable in the live defense.
"""
import gc, json, math, random, re, time
import torch
import torch.nn.functional as F

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


# --------------------------------------------------------------------------- data -> tensors
def build_example(row, tok):
    """Chat-template the system+user messages, append the assistant JSON + <|im_end|>; loss only on the completion.

    The prompt part is exactly what generate() receives at test time (add_generation_prompt=True), so training and
    inference see identical prefixes. Prompt and completion are tokenised separately; a check in the notebook shows
    this equals joint tokenisation on every row.
    """
    prompt = tok.apply_chat_template(row["messages"][:2], tokenize=False, add_generation_prompt=True)
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    c = tok(row["messages"][2]["content"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return {"id": row["id"], "input_ids": p + c, "labels": [-100] * len(p) + c, "n_prompt": len(p), "n_target": len(c)}


def collate(exs, pad_id, device):
    L = max(len(e["input_ids"]) for e in exs)
    ids = torch.full((len(exs), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(exs), L), dtype=torch.long)
    lab = torch.full((len(exs), L), -100, dtype=torch.long)
    for i, e in enumerate(exs):                      # right padding for training
        n = len(e["input_ids"])
        ids[i, :n] = torch.tensor(e["input_ids"])
        att[i, :n] = 1
        lab[i, :n] = torch.tensor(e["labels"])
    return {"input_ids": ids.to(device), "attention_mask": att.to(device), "labels": lab.to(device)}


def micro_batches(exs, budget):
    """Split one optimiser batch into micro-batches of similar length with <= `budget` padded tokens each."""
    exs = sorted(exs, key=lambda e: len(e["input_ids"]))
    out, cur = [], []
    for e in exs:
        if cur and (len(cur) + 1) * len(e["input_ids"]) > budget:
            out.append(cur)
            cur = []
        cur.append(e)
    if cur:
        out.append(cur)
    return out


def causal_lm(model):
    """The underlying *ForCausalLM: unwraps a PeftModel (LoRA layers live inside it); a plain HF model is returned as is."""
    try:
        from peft import PeftModel
    except ImportError:
        return model
    return model.base_model.model if isinstance(model, PeftModel) else model


def loss_sum(model, batch):
    """Sum of token cross-entropy over the assistant tokens (labels != -100) and how many tokens that is."""
    inner = causal_lm(model)
    h = inner.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state
    tgt = batch["labels"][:, 1:]                                                  # position t predicts token t+1
    keep = tgt != -100
    logits = inner.lm_head(h[:, :-1][keep]).float()                               # only the completion positions
    return F.cross_entropy(logits, tgt[keep], reduction="sum"), int(keep.sum())


# --------------------------------------------------------------------------- model
def _dtype_kw(transformers, dtype):
    ver = tuple(int(x) for x in transformers.__version__.split(".")[:2])
    return {"dtype" if ver >= (4, 56) else "torch_dtype": dtype}


def load_base(model_id=MODEL_ID, quant4=False):
    import transformers
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    kw = _dtype_kw(transformers, torch.float16)
    kw.update(attn_implementation="sdpa", device_map={"": 0})
    if quant4:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(model_id, **kw)


def add_lora(base, r=16, alpha=None, dropout=0.05, quant4=False, targets=LORA_TARGETS):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    gc_kw = {"use_reentrant": False}
    if quant4:
        base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True, gradient_checkpointing_kwargs=gc_kw)
    else:
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs=gc_kw)
        base.enable_input_require_grads()
    base.config.use_cache = False
    cfg = LoraConfig(r=r, lora_alpha=alpha or 2 * r, lora_dropout=dropout, target_modules=targets, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(base, cfg)
    for p in model.parameters():                       # fp32 adapters + fp16 autocast = stable mixed precision on a T4
        if p.requires_grad:
            p.data = p.data.float()
    return model


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    return tr, tot


def prepare_for_generation(model):
    inner = causal_lm(model)
    inner.config.use_cache = True
    model.eval()
    return model


# --------------------------------------------------------------------------- training
def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _grad_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def train(model, tok, examples, *, epochs=3, lr=2e-4, batch_size=16, warmup_ratio=0.05, micro_budget=4608,
          weight_decay=0.0, max_grad_norm=1.0, seed=0, amp=True, log_every=10, verbose=True):
    """AdamW + cosine schedule with warmup. One optimiser step = `batch_size` sequences, token-mean loss."""
    from transformers import get_cosine_schedule_with_warmup
    dev = next(model.parameters()).device
    use_amp = bool(amp and dev.type == "cuda")
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    steps_per_epoch = math.ceil(len(examples) / batch_size)
    total = steps_per_epoch * epochs
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(warmup_ratio * total)), total)
    scaler = _grad_scaler(use_amp)
    rng = random.Random(seed)
    model.train()
    hist, skipped, step, t0 = [], 0, 0, time.time()
    for ep in range(epochs):
        order = list(range(len(examples)))
        rng.shuffle(order)
        run = []
        for b in range(0, len(order), batch_size):
            batch = [examples[i] for i in order[b:b + batch_size]]
            n_tgt = sum(e["n_target"] for e in batch)
            tot_loss = 0.0
            for mb in micro_batches(batch, micro_budget):
                with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                    ls, _ = loss_sum(model, collate(mb, tok.pad_token_id, dev))
                scaler.scale(ls / n_tgt).backward()           # token-mean over the whole optimiser batch
                tot_loss += ls.item()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(params, max_grad_norm)
            before = scaler.get_scale() if use_amp else None
            scaler.step(opt)
            scaler.update()
            if use_amp and scaler.get_scale() < before:
                skipped += 1                                  # fp16 overflow -> step skipped by the scaler
            opt.zero_grad(set_to_none=True)
            sched.step()
            step += 1
            loss = tot_loss / n_tgt
            run.append(loss)
            if not math.isfinite(loss) and verbose:
                print(f"  !! non-finite loss at step {step}")
            if verbose and (step % log_every == 0 or step == total):
                print(f"  ep {ep + 1}/{epochs} step {step:4d}/{total} loss {sum(run[-log_every:]) / len(run[-log_every:]):.4f} "
                      f"lr {sched.get_last_lr()[0]:.2e} |g| {float(gn):.2f}  {(time.time() - t0) / 60:.1f} min")
            hist.append((step, loss))
    return {"loss_hist": hist, "seconds": time.time() - t0, "steps": step, "skipped_steps": skipped}


# --------------------------------------------------------------------------- generation
@torch.no_grad()
def generate(model, tok, rows, *, batch_size=16, max_new_tokens=200, system_override=None, **gen_kwargs):
    """Greedy decoding exactly as in the assignment: chat template on [system, user], do_sample=False, 200 new tokens.

    Nothing else is changed: the model's own generation_config (incl. its repetition_penalty) stays in force unless the
    caller passes an override, which is how the sensitivity check in the notebook works.
    Returns the raw decoded text with only .strip() applied.
    """
    prepare_for_generation(model)
    dev = next(model.parameters()).device
    old_side = tok.padding_side
    tok.padding_side = "left"
    prompts = []
    for r in rows:
        msgs = r["messages"][:2]
        if system_override is not None:
            msgs = [{"role": "system", "content": system_override}, msgs[1]]
        prompts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    order = sorted(range(len(rows)), key=lambda i: -len(prompts[i]))      # similar lengths together -> little padding
    outs = [None] * len(rows)
    for b in range(0, len(order), batch_size):
        idx = order[b:b + batch_size]
        enc = tok([prompts[i] for i in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(dev)
        out = model.generate(**enc, do_sample=False, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **gen_kwargs)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        for i, t in zip(idx, texts):
            outs[i] = t.strip()
    tok.padding_side = old_side
    return outs


def lenient_json(text):
    """Diagnostic only (never used for the official score): strip code fences / prose around the first {...}."""
    if text is None:
        return None
    t = re.sub(r"```(?:json)?", "", text).strip()
    m = re.search(r"\{.*\}", t, re.S)
    return m.group(0) if m else t


# --------------------------------------------------------------------------- bookkeeping
def peak_gb():
    return torch.cuda.max_memory_allocated() / 2 ** 30 if torch.cuda.is_available() else float("nan")


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Writing /content/nimbus/nimbus_train.py


In [ ]:
sys.path.insert(0, "/content/nimbus"); sys.path.insert(0, PACK)
import nimbus_lib as nl, nimbus_clean as nc, nimbus_train as nt, score as scorer
nl.ROOT = f"{PACK}/data/"
raw, dev, test = nl.load("train.jsonl"), nl.load("dev.jsonl"), nl.load("test_inputs.jsonl")
print("rows: train", len(raw), "| dev", len(dev), "| test", len(test))
FIXED_SYSTEM = raw[0]["messages"][0]["content"]
assert all(r["messages"][0]["content"] == FIXED_SYSTEM for r in raw + dev + test)
print("system prompt:", FIXED_SYSTEM)

rows: train 1445 | dev 200 | test 400
system prompt: You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.


## Part A: audit and clean the training data

`train.jsonl` is not clean. The audit combines (1) a structural profile, (2) sentence-template mining (the tickets are generated from a fixed phrase inventory), (3) a rule checker that re-derives every deterministic field from the ticket text using only SCHEMA.md, judging each field against the *derived truth* rather than against the other label fields, (4) a cross-validated category check, and (5) exact-duplicate / leakage checks. The full scripts and write-up are in `audit/` in the repository; the cells below reproduce the headline numbers and run the cleaning.

In [ ]:
# How problems 1-2 were found: structural profile
bad_json = [r["id"] for r in raw if nl.parse_label(r["messages"][2]["content"]) is None]
key_sets = collections.Counter(tuple(k for k in (nl.parse_label(r["messages"][2]["content"]) or {"<unparseable>": 0}))[:2] for r in raw)
print("labels that are not valid JSON (truncated):", len(bad_json), bad_json[:4], "...")
print("label key sets (first 2 keys):", dict(key_sets))
legacy = [r for r in raw if '"prio"' in r["messages"][2]["content"]]
print("legacy v1 rows:", len(legacy), "| example:", legacy[0]["messages"][2]["content"][:140])

labels that are not valid JSON (truncated): 25 ['tr-00060', 'tr-00095', 'tr-00125', 'tr-00172'] ...
label key sets (first 2 keys): {('category', 'priority'): 1350, ('category', 'prio'): 70, ('<unparseable>',): 25}
legacy v1 rows: 70 | example: {"category": "refund", "prio": "low", "amount_paise": 97800, "txn_date": "02/09/2026", "txn_id": null, "channel": "wallet", "language": "en"


In [ ]:
clean, log, stats = nc.clean_dataset(raw, dev)
print("input", len(raw), "-> kept", len(clean), "| dropped", sum(a == "drop" for _, a, _ in log), "| fixed", sum(a == "fix" for _, a, _ in log))
print(dict(stats))
assert stats["drop_unexplained"] == 0, "every non-conforming row must match a known corruption signature"
with open(f"{WORK}/cleaning_log.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["id", "action", "reason"]); w.writerows(log)
with open(f"{WORK}/train_clean.jsonl", "w", encoding="utf-8") as f:
    for r in clean:
        f.write(json.dumps(r, ensure_ascii=False) + chr(10))
pd.DataFrame(log, columns=["id", "action", "reason"]).assign(kind=lambda d: d.reason.str.replace(r"[\d]+", "#", regex=True).str[:55]).groupby(["action", "kind"]).size().rename("rows").reset_index()

input 1445 -> kept 1320 | dropped 125 | fixed 175
{'drop_dev_leak': 30, 'fix_priority': 60, 'fix_legacy': 70, 'drop_junk': 10, 'drop_truncated': 25, 'fix_txn_id': 20, 'fix_amount': 25, 'drop_duplicate': 60}


,action,kind,rows
0,drop,assistant label is truncated / invalid JSON,25
1,drop,exact duplicate of tr-# (same text and label),60
2,drop,junk ticket (body=''): label has fields the te...,1
3,drop,junk ticket (body='asdf asdf'): label has fiel...,3
4,drop,junk ticket (body='test test #'): label has fi...,3
5,drop,junk ticket (body='test'): label has fields th...,3
6,drop,leakage: ticket text identical to dev ticket d...,30
7,fix,amount_paise stored in rupees (#) instead of p...,25
8,fix,legacy v# format: 'prio':'high'->'priority':'P...,11
9,fix,legacy v# format: 'prio':'low'->'priority':'P#...,44


In [ ]:
# Evidence that it is the *priority* column that is corrupted (problem 8): needs_human always follows the spec-derived priority
pri_fixed = [i for i, a, why in log if a == "fix" and "contradicts spec" in why]
by_id = {r["id"]: r for r in raw}
agree_derived = informative = informative_agree = 0
for i in pri_fixed:
    lab = nl.parse_label(by_id[i]["messages"][2]["content"])
    lab = {("priority" if k == "prio" else k): v for k, v in lab.items()}
    d = nl.derive(by_id[i]["messages"][1]["content"], lab["category"])
    agree_derived += lab["needs_human"] == d["needs_human"]
    if nl.exp_needs_human(lab["priority"], d["_contacted"]) != d["needs_human"]:   # rows where the bad priority WOULD imply another needs_human
        informative += 1
        informative_agree += lab["needs_human"] == d["needs_human"]
print(f"priority-fixed rows: {len(pri_fixed)} | needs_human equals the spec-derived value in {agree_derived}/{len(pri_fixed)}")
print(f"in the {informative} rows where the corrupted priority would imply a different needs_human, the label agrees with the derived priority in {informative_agree}/{informative} (and with the corrupted one in {informative - informative_agree})")

# Validation of the checker itself: derive() vs the dev gold labels (dev used for MEASUREMENT only, never to decide a fix)
fields = ["priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
hit = collections.Counter()
for r in dev:
    gold = json.loads(r["messages"][2]["content"]); d = nl.derive(r["messages"][1]["content"], gold["category"])
    for k in fields:
        hit[k] += d[k] == gold[k]
print("derive() vs dev gold:", {k: f"{hit[k] / len(dev):.1%}" for k in fields})
DEV_LEAK = {r["id"] for r in dev if r["messages"][1]["content"] in {x["messages"][1]["content"] for x in raw}}
print("dev tickets that also appear in raw train:", len(DEV_LEAK))

priority-fixed rows: 60 | needs_human equals the spec-derived value in 60/60
in the 43 rows where the corrupted priority would imply a different needs_human, the label agrees with the derived priority in 43/43 (and with the corrupted one in 0)
derive() vs dev gold: {'priority': '100.0%', 'amount_paise': '100.0%', 'txn_date': '100.0%', 'txn_id': '100.0%', 'channel': '100.0%', 'language': '100.0%', 'needs_human': '100.0%'}
dev tickets that also appear in raw train: 30


## Evaluation helpers
Scoring uses the provided `score.py` untouched. Besides the official dev numbers, every run also reports `exact_clean_dev`: exact match on the 170 dev tickets that do **not** appear in raw train, so the leakage in the raw file cannot flatter a run.

In [ ]:
RESULTS = f"{WORK}/logs/results.jsonl"


def evaluate(gold_rows, texts, tag=None):
    preds = [{"id": g["id"], "output": t} for g, t in zip(gold_rows, texts)]
    if tag:
        with open(f"{WORK}/preds/{tag}.jsonl", "w", encoding="utf-8") as f:
            for p in preds:
                f.write(json.dumps(p, ensure_ascii=False) + chr(10))
    res = scorer.score(gold_rows, preds)
    res_clean = scorer.score([g for g in gold_rows if g["id"] not in DEV_LEAK], preds)
    a = res["all"]
    return {"exact": a["exact_match"], "field_acc": a["mean_field_acc"], "json_valid": a["json_valid"],
            "exact_clean_dev": res_clean["all"]["exact_match"], "fields": a["fields"], "by_category": {k: v["exact_match"] for k, v in res.items() if k.startswith("category=")}}


def log_result(entry):
    with open(RESULTS, "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + chr(10))


def results_table():
    rows = [json.loads(l) for l in open(RESULTS, encoding="utf-8")] if os.path.exists(RESULTS) else []
    last = {r["name"]: r for r in rows if not r["name"].endswith("__train_only")}   # latest entry per run name
    cols = ["name", "desc", "n_train", "trainable_M", "exact", "field_acc", "json_valid", "exact_clean_dev", "exact_rep1.0", "peak_gb", "train_min"]
    df = pd.DataFrame(list(last.values())).reindex(columns=cols)
    return df.round(3)

## Part B: baselines on dev (no fine-tuning)

1. **B1:** base model, the fixed system prompt + ticket (exactly what the fine-tuned model will get).
2. **B2:** same model with the full SCHEMA.md rules written into the system prompt (allowed here only, as a reference point).

Greedy decoding, `max_new_tokens=200`, the model's own chat template, strict scoring (the output must be *only* a JSON object). I also print a lenient score (code fences / prose stripped) as a diagnostic, never as the result.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(nt.MODEL_ID)
print("eos:", tok.eos_token, "| pad:", tok.pad_token, "| padding_side:", tok.padding_side)

torch.cuda.reset_peak_memory_stats()
base = nt.load_base(quant4=False)
print("base model loaded in fp16, params:", round(sum(p.numel() for p in base.parameters()) / 1e9, 2), "B | GPU GB:", round(nt.peak_gb(), 2))
print("model generation_config:", base.generation_config)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

eos: <|im_end|> | pad: <|endoftext|> | padding_side: right


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

base model loaded in fp16, params: 1.54 B | GPU GB: 2.88
model generation_config: GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}



In [ ]:
# B1: fixed prompt
t0 = time.time(); out_b1 = nt.generate(base, tok, dev, batch_size=16); m1 = (time.time() - t0) / 60
r_b1 = evaluate(dev, out_b1, "B1_base_fixed_prompt_dev")
lenient = evaluate(dev, [nt.lenient_json(t) for t in out_b1])
print(f"B1 strict: exact {r_b1['exact']:.1%} | field acc {r_b1['field_acc']:.1%} | valid JSON {r_b1['json_valid']:.1%}   ({m1:.1f} min)")
print(f"B1 lenient (diagnostic only): exact {lenient['exact']:.1%} | field acc {lenient['field_acc']:.1%} | valid {lenient['json_valid']:.1%}")
for i in (0, 1, 2):
    print("--- raw output", i, "---"); print(out_b1[i][:400])
log_result({"name": "B1_base_fixed_prompt", "desc": "base model, fixed prompt, no fine-tuning", "n_train": 0, "trainable_M": 0, "exact": r_b1["exact"], "field_acc": r_b1["field_acc"],
            "json_valid": r_b1["json_valid"], "exact_clean_dev": r_b1["exact_clean_dev"], "peak_gb": nt.peak_gb(), "train_min": 0})

B1 strict: exact 0.0% | field acc 0.0% | valid JSON 0.0%   (2.0 min)
B1 lenient (diagnostic only): exact 0.0% | field acc 0.0% | valid 98.0%
--- raw output 0 ---
```json
{
  "status": "New",
  "priority": "High",
  "category": "Card Activation",
  "description": "User received an activation request for their card but has not yet been able to use it. They need assistance with activating and using the card.",
  "ticket_number": "#54047"
}
```
--- raw output 1 ---
```json
{
  "priority": "normal",
  "category": "customer_service",
  "description": "Customer inquires about accepted forms of identification for transactions on their Realme 11 device.",
  "resolution_required": true,
  "expected_resolution_timeframe": "within 3 business days"
}
```
--- raw output 2 ---
```json
{
  "status": "urgent",
  "priority": "high",
  "ticket_id": "TICKET_123456",
  "customer_name": "Customer Name",
  "contact_method": "phone",
  "source": "ivr-transcript",
  "message": "Dear Customer, we understand you

In [ ]:
# B2: rules in the prompt (reference only)
RULES = open(f"{PACK}/SCHEMA.md", encoding="utf-8").read()
SYSTEM_RULES = FIXED_SYSTEM + "\n\n" + RULES + "\n\nReply with the JSON object only, no explanation and no code fences."
t0 = time.time(); out_b2 = nt.generate(base, tok, dev, batch_size=4, system_override=SYSTEM_RULES); m2 = (time.time() - t0) / 60
r_b2 = evaluate(dev, out_b2, "B2_base_rules_in_prompt_dev")
lenient2 = evaluate(dev, [nt.lenient_json(t) for t in out_b2])
print(f"B2 strict: exact {r_b2['exact']:.1%} | field acc {r_b2['field_acc']:.1%} | valid JSON {r_b2['json_valid']:.1%}   ({m2:.1f} min)")
print(f"B2 lenient (diagnostic only): exact {lenient2['exact']:.1%} | field acc {lenient2['field_acc']:.1%} | valid {lenient2['json_valid']:.1%}")
print({k: round(v, 3) for k, v in r_b2["fields"].items()})
print("--- raw output 0 ---"); print(out_b2[0][:400])
log_result({"name": "B2_base_rules_in_prompt", "desc": "base model, SCHEMA.md rules in system prompt (reference)", "n_train": 0, "trainable_M": 0, "exact": r_b2["exact"], "field_acc": r_b2["field_acc"],
            "json_valid": r_b2["json_valid"], "exact_clean_dev": r_b2["exact_clean_dev"], "peak_gb": nt.peak_gb(), "train_min": 0})
del base; nt.free_memory()

B2 strict: exact 0.0% | field acc 19.4% | valid JSON 34.0%   (4.0 min)
B2 lenient (diagnostic only): exact 0.5% | field acc 59.9% | valid 100.0%
{'category': 0.235, 'priority': 0.085, 'amount_paise': 0.275, 'txn_date': 0.225, 'txn_id': 0.225, 'channel': 0.19, 'language': 0.23, 'needs_human': 0.085}
--- raw output 0 ---
```json
{
  "category": "kyc",
  "priority": "P3",
  "amount_paise": null,
  "txn_date": null,
  "txn_id": null,
  "channel": null,
  "language": "en",
  "needs_human": true
}
```


## Part C: the fine-tuning pipeline

Evidence for each design choice (template, loss tokens, sequence length, parameter budget) is printed below.

**Chat template.** The prompt is built with the model's own template on `[system, user]` plus `add_generation_prompt=True`: this is exactly what `generate` receives at test time. The target is the assistant JSON followed by `<|im_end|>`, so the model learns to stop (the output must be *only* the JSON).

**Loss on the right tokens.** Prompt tokens are masked; the loss is token cross-entropy on the assistant JSON and `<|im_end|>` only. It is computed only at those positions (the 152k-vocabulary logits are never built for prompt tokens).

In [ ]:
ex = clean[1]
full = tok.apply_chat_template(ex["messages"], tokenize=False)
prompt = tok.apply_chat_template(ex["messages"][:2], tokenize=False, add_generation_prompt=True)
print(full)
assert full.startswith(prompt), "training prompt must be a prefix of the full chat"
e = nt.build_example(ex, tok)
print("\nprompt tokens:", e["n_prompt"], "| supervised tokens:", e["n_target"])
print("supervised text:", repr(tok.decode([t for t in e["labels"] if t != -100])))
print("masked prompt tokens contribute no loss:", all(l == -100 for l in e["labels"][:e["n_prompt"]]))
def joint_ids(r):   # tokenise prompt + completion as ONE string
    s = tok.apply_chat_template(r["messages"][:2], tokenize=False, add_generation_prompt=True) + r["messages"][2]["content"] + tok.eos_token
    return tok(s, add_special_tokens=False)["input_ids"]
joint_mismatch = sum(joint_ids(r) != nt.build_example(r, tok)["input_ids"] for r in clean)
print("rows where separate prompt/completion tokenisation differs from joint tokenisation:", joint_mismatch, "of", len(clean))
print("numbers are tokenised digit by digit:", tok.tokenize('"amount_paise": 4750000'))

<|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-06-15 (Mon)
Source: twitter

@NimbusPayCare Where can I reset the passcode?<|im_end|>
<|im_start|>assistant
{"category": "account_access", "priority": "P2", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}<|im_end|>


prompt tokens: 66 | supervised tokens: 51
supervised text: '{"category": "account_access", "priority": "P2", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language": "en", "needs_human": false}<|im_end|>'
masked prompt tokens contribute no loss: True
rows where separate prompt/completion tokenisation differs from joint tokenisation: 0 of 1320
numbers are tokenised digit by digit: ['"', 'amount', '_p', 'aise', '":', 'Ġ', '4', '7', '5', '0', '0', '0', '0']


In [ ]:
# Sequence length chosen from evidence
exs_clean = [nt.build_example(r, tok) for r in clean]
L = np.array([len(e["input_ids"]) for e in exs_clean]); T = np.array([e["n_target"] for e in exs_clean])
print("total tokens per example percentiles:", {p: int(np.percentile(L, p)) for p in (50, 90, 95, 97, 99, 100)})
print("assistant tokens per example (p50/p99/max):", int(np.median(T)), int(np.percentile(T, 99)), int(T.max()))
for cap in (320, 512, 1024, 2048, 2560):
    print(f"  cap {cap:5d}: {int((L > cap).sum()):4d} examples ({(L > cap).mean():.1%}) would be cut")
print("examples longer than 512 tokens all contain a quoted '>' thread:", all(">" in clean[i]["messages"][1]["content"] for i in np.where(L > 512)[0]))
MAX_LEN = 2560
assert L.max() <= MAX_LEN
print("-> MAX_LEN =", MAX_LEN, f"(longest example {L.max()} tokens; nothing truncated)")

total tokens per example percentiles: {50: 167, 90: 213, 95: 225, 97: 263, 99: 1912, 100: 2214}
assistant tokens per example (p50/p99/max): 65 82 84
  cap   320:   38 examples (2.9%) would be cut
  cap   512:   38 examples (2.9%) would be cut
  cap  1024:   38 examples (2.9%) would be cut
  cap  2048:    9 examples (0.7%) would be cut
  cap  2560:    0 examples (0.0%) would be cut
examples longer than 512 tokens all contain a quoted '>' thread: True
-> MAX_LEN = 2560 (longest example 2214 tokens; nothing truncated)


**Sequence length.** The distribution is bimodal: 97% of examples are at most ~260 tokens, and the other ~3% are tickets with a quoted earlier-email thread (1,000-2,200 tokens). The label sits at the *end* of the sequence, so any truncation would delete the target. Dropping those tickets would also remove the only examples that teach the spec rule *"dates inside quoted earlier emails are not the transaction date"*. So: no truncation, `MAX_LEN = 2560` (longest example rounded up). Memory is bounded by token-budget micro-batching (<= 4,608 padded tokens per forward pass, similar lengths grouped), gradient checkpointing, and computing logits only on assistant positions.

**Parameter budget.** LoRA on all linear layers (attention q,k,v,o and MLP gate,up,down): the rules need arithmetic/format conversion, which lives in the MLPs as much as in attention. The cap is 50M trainable parameters:

In [ ]:
cfg = transformers.AutoConfig.from_pretrained(nt.MODEL_ID)
H, I, NL = cfg.hidden_size, cfg.intermediate_size, cfg.num_hidden_layers
kv = cfg.num_key_value_heads * (H // cfg.num_attention_heads)
dims = {"q_proj": (H, H), "k_proj": (H, kv), "v_proj": (H, kv), "o_proj": (H, H), "gate_proj": (H, I), "up_proj": (H, I), "down_proj": (I, H)}
for r_ in (8, 16, 32, 64):
    n = NL * sum(r_ * (a + b) for a, b in dims.values())
    print(f"rank {r_:2d}: {n / 1e6:6.2f}M trainable parameters  {'<= 50M OK' if n <= 50e6 else '> 50M NOT ALLOWED'}")

rank  8:   9.23M trainable parameters  <= 50M OK
rank 16:  18.46M trainable parameters  <= 50M OK
rank 32:  36.93M trainable parameters  <= 50M OK
rank 64:  73.86M trainable parameters  > 50M NOT ALLOWED


**Default configuration (run R0), with reasons**

| Setting | Value | Why |
|---|---|---|
| Base model | Qwen2.5-1.5B-Instruct, fp16 | <= 4B; chat template has a system role; fits a T4 in 16-bit *and* 4-bit so the LoRA/QLoRA comparison is fair |
| LoRA | r = 16, alpha = 32, dropout 0.05, all linear layers | 18.5M params (cap 50M); alpha = 2r is the usual scale; r = 64 would break the cap |
| Optimiser | AdamW, lr 2e-4, cosine decay, 5% warmup, no weight decay | standard LoRA learning rate; varied later in Part D |
| Batch | 16 sequences per step (token-mean loss) | ~83 steps/epoch on 1,320 rows |
| Epochs | 3 | ~250 steps; varied later in Part D |
| Precision | fp16 autocast + GradScaler, fp32 LoRA weights | T4 has no bf16; fp32 adapters keep the update stable |
| Memory | gradient checkpointing, micro-batch budget 4,608 tokens | needed for the 2,200-token quoted-thread tickets |
| Decoding | greedy, `max_new_tokens=200`, model's own generation_config untouched | exactly as the assignment specifies |

In [ ]:
def run_experiment(name, train_rows, *, desc, quant4=False, r=16, alpha=None, lr=2e-4, epochs=3, dropout=0.05, seed=0, batch_size=16, sens=True):
    nt.free_memory()
    # a model left over from an earlier cell would inflate the peak-memory number and can cause OOM at evaluation time
    assert torch.cuda.memory_allocated() < 0.5 * 2**30, f"{torch.cuda.memory_allocated() / 2**30:.1f} GB still allocated: run  del model; nt.free_memory()  first"
    torch.cuda.reset_peak_memory_stats(); nt.seed_everything(seed)
    base = nt.load_base(quant4=quant4)
    model = nt.add_lora(base, r=r, alpha=alpha, dropout=dropout, quant4=quant4)
    trainable, total = nt.count_params(model)
    exs = [nt.build_example(x, tok) for x in train_rows]
    assert max(len(e["input_ids"]) for e in exs) <= MAX_LEN
    print(f"[{name}] {desc}\n  rows {len(exs)} | trainable {trainable / 1e6:.2f}M of {total / 1e6:.0f}M | quant4={quant4} r={r} lr={lr} epochs={epochs}")
    res = nt.train(model, tok, exs, epochs=epochs, lr=lr, batch_size=batch_size, seed=seed)
    train_min, peak = res["seconds"] / 60, nt.peak_gb()
    model.save_pretrained(f"{WORK}/adapters/{name}")
    log_result({"name": name + "__train_only", "desc": desc, "n_train": len(exs), "trainable_M": trainable / 1e6, "peak_gb": peak, "train_min": train_min})   # survives an evaluation crash
    print(f"  training done: {train_min:.1f} min, peak {peak:.1f} GB, adapter saved")
    t0 = time.time(); outs = nt.generate(model, tok, dev, batch_size=16); eval_min = (time.time() - t0) / 60
    ev = evaluate(dev, outs, f"{name}_dev")
    entry = {"name": name, "desc": desc, "n_train": len(exs), "trainable_M": trainable / 1e6, "quant4": quant4, "r": r, "lr": lr, "epochs": epochs,
             "exact": ev["exact"], "field_acc": ev["field_acc"], "json_valid": ev["json_valid"], "exact_clean_dev": ev["exact_clean_dev"],
             "fields": ev["fields"], "by_category": ev["by_category"], "peak_gb": peak, "train_min": train_min, "eval_min": eval_min,
             "skipped_steps": res["skipped_steps"], "final_loss": float(np.mean([l for _, l in res["loss_hist"][-10:]]))}
    if sens:   # sensitivity: the model's generation_config has repetition_penalty=1.1, which stays active in the default run
        ev2 = evaluate(dev, nt.generate(model, tok, dev, batch_size=16, repetition_penalty=1.0))
        entry["exact_rep1.0"], entry["field_acc_rep1.0"] = ev2["exact"], ev2["field_acc"]
    log_result(entry)
    print(f"  -> exact {ev['exact']:.1%} | field acc {ev['field_acc']:.1%} | valid {ev['json_valid']:.1%} | exact on clean dev {ev['exact_clean_dev']:.1%}"
          + (f" | exact with repetition_penalty=1.0: {entry['exact_rep1.0']:.1%}" if sens else "")
          + f" | peak {peak:.1f} GB | train {train_min:.1f} min")
    return model, entry

### R0: reference run (cleaned data, 16-bit LoRA)

In [ ]:
m, e0 = run_experiment("R0_clean_lora16", clean, desc="cleaned data, 16-bit LoRA r=16, lr 2e-4, 3 epochs")
del m; nt.free_memory()   # the adapter is saved to Drive; do not keep the model on the GPU

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[R0_clean_lora16] cleaned data, 16-bit LoRA r=16, lr 2e-4, 3 epochs
  rows 1320 | trainable 18.46M of 1562M | quant4=False r=16 lr=0.0002 epochs=3
  ep 1/3 step   10/249 loss 1.4580 lr 1.67e-04 |g| 2.21  0.6 min
  ep 1/3 step   20/249 loss 0.1590 lr 1.99e-04 |g| 0.33  1.1 min
  ep 1/3 step   30/249 loss 0.0532 lr 1.97e-04 |g| 0.36  1.7 min
  ep 1/3 step   40/249 loss 0.0451 lr 1.93e-04 |g| 0.39  2.4 min
  ep 1/3 step   50/249 loss 0.0344 lr 1.88e-04 |g| 0.22  3.0 min
  ep 1/3 step   60/249 loss 0.0326 lr 1.80e-04 |g| 0.22  3.5 min
  ep 1/3 step   70/249 loss 0.0295 lr 1.72e-04 |g| 0.18  4.1 min
  ep 1/3 step   80/249 loss 0.0195 lr 1.62e-04 |g| 0.20  4.7 min
  ep 2/3 step   90/249 loss 0.0147 lr 1.51e-04 |g| 0.49  5.3 min
  ep 2/3 step  100/249 loss 0.0138 lr 1.39e-04 |g| 0.20  5.8 min
  ep 2/3 step  110/249 loss 0.0093 lr 1.27e-04 |g| 0.13  6.6 min
  ep 2/3 step  120/249 loss 0.0110 lr 1.14e-04 |g| 0.17  7.1 min
  ep 2/3 step  130/249 loss 0.0069 lr 1.01e-04 |g| 0.08  7.7 min
  ep 2/3

In [ ]:
print({k: round(v, 3) for k, v in e0["fields"].items()})
print("exact match by category:", {k.split("=")[1]: round(v, 2) for k, v in e0["by_category"].items()})
results_table()

{'category': 0.965, 'priority': 0.92, 'amount_paise': 0.995, 'txn_date': 0.99, 'txn_id': 1.0, 'channel': 0.995, 'language': 1.0, 'needs_human': 0.96}
exact match by category: {'other': 0.94, 'kyc': 1.0, 'refund': 0.82, 'payment_failed': 0.85, 'fraud': 0.8, 'offers': 1.0, 'account_access': 1.0}


,name,desc,n_train,trainable_M,exact,field_acc,json_valid,exact_clean_dev,exact_rep1.0,peak_gb,train_min
0,B1_base_fixed_prompt,"base model, fixed prompt, no fine-tuning",0,0.000,0.00,0.000,0.00,0.000,NaN,5.954,0.000
1,B2_base_rules_in_prompt,"base model, SCHEMA.md rules in system prompt (...",0,0.000,0.00,0.194,0.34,0.000,NaN,5.954,0.000
2,R0_clean_lora16,"cleaned data, 16-bit LoRA r=16, lr 2e-4, 3 epochs",1320,18.465,0.89,0.978,1.00,0.888,0.900,5.334,14.544
3,D1_raw_lora16,"RAW train.jsonl (1,445 rows, uncleaned), other...",1445,18.465,0.79,0.965,1.00,0.776,0.870,5.308,15.438
4,D2_clean_qlora4,"cleaned data, 4-bit QLoRA (NF4, double quant, ...",1320,18.465,0.88,0.977,1.00,0.882,0.905,4.700,17.666
5,D4_clean_lora16_keyorder,"cleaned data, derived fields (priority, needs_...",1320,18.465,0.87,0.975,1.00,0.876,0.890,5.335,14.292
6,D3a_clean_lora16_1epoch,"cleaned data, 16-bit LoRA, 1 epoch (R0: 3)",1320,18.465,0.42,0.901,1.00,0.424,0.480,5.265,4.774
7,D3b_clean_lora16_5epochs,"cleaned data, 16-bit LoRA, 5 epochs (R0: 3)",1320,18.465,0.93,0.986,1.00,0.924,0.930,5.379,23.783


## Part D: controlled experiments

Every run below changes **exactly one thing** relative to R0 (cleaned data, 16-bit LoRA r=16, lr 2e-4, 3 epochs, seed 0) and is scored on the same 200 dev tickets. Each run logs exact match, mean field accuracy, peak GPU memory (training), training minutes, and exact match with `repetition_penalty=1.0` (the model's default generation_config applies 1.1).

**How this section runs.** Re-running all experiments takes about 2 hours on a T4, which would break the 3-hour limit for the whole notebook. So by default (`RUN_EXPERIMENTS = False`) the experiment cells are skipped and the table is rebuilt from the log of my original Colab runs (embedded below, produced by exactly the code in these cells). Set `RUN_EXPERIMENTS = True` to re-run everything. R0 above and the final model below are always run live, which also checks that the saved R0 numbers reproduce.

In [ ]:
# Log of my original Part D runs (Colab T4, seed 0): one JSON line per run, written by run_experiment()
SAVED_LOG = r"""
{"name": "B1_base_fixed_prompt", "desc": "base model, fixed prompt, no fine-tuning", "n_train": 0, "trainable_M": 0.0, "exact": 0.0, "field_acc": 0.0, "json_valid": 0.0, "exact_clean_dev": 0.0, "peak_gb": 5.954000949859619, "train_min": 0.0, "source": "Colab T4 (Tesla T4, 14.6 GB), seed 0, same 200 dev tickets; values copied from results.jsonl on 2026-11-12"}
{"name": "B2_base_rules_in_prompt", "desc": "base model, SCHEMA.md rules in system prompt (reference)", "n_train": 0, "trainable_M": 0.0, "exact": 0.0, "field_acc": 0.19375, "json_valid": 0.34, "exact_clean_dev": 0.0, "peak_gb": 5.954000949859619, "train_min": 0.0, "source": "Colab T4 (Tesla T4, 14.6 GB), seed 0, same 200 dev tickets; values copied from results.jsonl on 2026-11-12"}
{"name": "R0_clean_lora16", "desc": "cleaned data, 16-bit LoRA r=16, lr 2e-4, 3 epochs", "n_train": 1320, "trainable_M": 18.464768, "exact": 0.89, "field_acc": 0.97875, "json_valid": 1.0, "exact_clean_dev": 0.888235294117647, "exact_rep1.0": 0.905, "peak_gb": 8.209423542022705, "train_min": 14.298393233617146, "eval_min": 1.6528451244036357, "note": "peak 8.2 GB is inflated by ~3 GB of GPU memory leaked from a failed earlier attempt in the same session; clean 16-bit runs with identical shapes peak at 5.3 GB (D1, D3a, D3b, D4)", "source": "Colab T4 (Tesla T4, 14.6 GB), seed 0, same 200 dev tickets; values copied from results.jsonl on 2026-11-12"}
{"name": "D1_raw_lora16", "desc": "RAW train.jsonl (1,445 rows, uncleaned), otherwise as R0", "n_train": 1445, "trainable_M": 18.464768, "exact": 0.79, "field_acc": 0.965, "json_valid": 1.0, "exact_clean_dev": 0.7764705882352941, "exact_rep1.0": 0.87, "peak_gb": 5.308108329772949, "train_min": 15.437685020764668, "eval_min": 1.7324058175086976, "source": "Colab T4 (Tesla T4, 14.6 GB), seed 0, same 200 dev tickets; values copied from results.jsonl on 2026-11-12"}
{"name": "D2_clean_qlora4", "desc": "cleaned data, 4-bit QLoRA (NF4, double quant, fp16 compute), otherwise as R0", "n_train": 1320, "trainable_M": 18.464768, "exact": 0.88, "field_acc": 0.9768749999999999, "json_valid": 1.0, "exact_clean_dev": 0.8823529411764706, "exact_rep1.0": 0.905, "peak_gb": 4.700083255767822, "train_min": 17.665899364153542, "eval_min": 2.378486967086792, "source": "Colab T4 (Tesla T4, 14.6 GB), seed 0, same 200 dev tickets; values copied from results.jsonl on 2026-11-12"}
{"name": "D3a_clean_lora16_1epoch", "desc": "cleaned data, 16-bit LoRA, 1 epoch (R0: 3)", "n_train": 1320, "trainable_M": 18.464768, "exact": 0.42, "field_acc": 0.900625, "json_valid": 1.0, "exact_clean_dev": 0.4235294117647059, "exact_rep1.0": 0.48, "peak_gb": 5.265270709991455, "train_min": 4.773935349782308, "eval_min": 1.6862078229586284, "source": "Colab T4 (Tesla T4, 14.6 GB), seed 0, same 200 dev tickets; values copied from results.jsonl on 2026-11-12"}
{"name": "D3b_clean_lora16_5epochs", "desc": "cleaned data, 16-bit LoRA, 5 epochs (R0: 3)", "n_train": 1320, "trainable_M": 18.464768, "exact": 0.93, "field_acc": 0.98625, "json_valid": 1.0, "exact_clean_dev": 0.9235294117647059, "exact_rep1.0": 0.93, "peak_gb": 5.379208564758301, "train_min": 23.78276682694753, "eval_min": 1.6711050470670064, "source": "Colab T4 (Tesla T4, 14.6 GB), seed 0, same 200 dev tickets; values copied from results.jsonl on 2026-11-12"}
{"name": "D4_clean_lora16_keyorder", "desc": "cleaned data, derived fields (priority, needs_human) emitted last, otherwise as R0", "n_train": 1320, "trainable_M": 18.464768, "exact": 0.87, "field_acc": 0.975, "json_valid": 1.0, "exact_clean_dev": 0.8764705882352941, "exact_rep1.0": 0.89, "peak_gb": 5.334794521331787, "train_min": 14.291571931044261, "eval_min": 1.693821334838867, "source": "Colab T4 (Tesla T4, 14.6 GB), seed 0, same 200 dev tickets; values copied from results.jsonl on 2026-11-12"}
"""

In [ ]:
RUN_EXPERIMENTS = False     # True: re-run every Part D experiment (about 2 h on a T4)

def saved_table():
    rows = [json.loads(l) for l in SAVED_LOG.strip().splitlines()]
    cols = ["name", "desc", "n_train", "trainable_M", "exact", "field_acc", "json_valid", "exact_clean_dev", "exact_rep1.0", "peak_gb", "train_min"]
    return pd.DataFrame(rows).reindex(columns=cols).round(3)

def run_or_skip(name, rows, **kw):
    if RUN_EXPERIMENTS:
        m, e = run_experiment(name, rows, **kw)
        del m; nt.free_memory()
        return e
    print(f"RUN_EXPERIMENTS=False: {name} not re-run, its result is in the saved log table")

saved_table()

,name,desc,n_train,trainable_M,exact,field_acc,json_valid,exact_clean_dev,exact_rep1.0,peak_gb,train_min
0,B1_base_fixed_prompt,"base model, fixed prompt, no fine-tuning",0,0.000,0.00,0.000,0.00,0.000,NaN,5.954,0.000
1,B2_base_rules_in_prompt,"base model, SCHEMA.md rules in system prompt (...",0,0.000,0.00,0.194,0.34,0.000,NaN,5.954,0.000
2,R0_clean_lora16,"cleaned data, 16-bit LoRA r=16, lr 2e-4, 3 epochs",1320,18.465,0.89,0.979,1.00,0.888,0.905,8.209,14.298
3,D1_raw_lora16,"RAW train.jsonl (1,445 rows, uncleaned), other...",1445,18.465,0.79,0.965,1.00,0.776,0.870,5.308,15.438
4,D2_clean_qlora4,"cleaned data, 4-bit QLoRA (NF4, double quant, ...",1320,18.465,0.88,0.977,1.00,0.882,0.905,4.700,17.666
5,D3a_clean_lora16_1epoch,"cleaned data, 16-bit LoRA, 1 epoch (R0: 3)",1320,18.465,0.42,0.901,1.00,0.424,0.480,5.265,4.774
6,D3b_clean_lora16_5epochs,"cleaned data, 16-bit LoRA, 5 epochs (R0: 3)",1320,18.465,0.93,0.986,1.00,0.924,0.930,5.379,23.783
7,D4_clean_lora16_keyorder,"cleaned data, derived fields (priority, needs_...",1320,18.465,0.87,0.975,1.00,0.876,0.890,5.335,14.292


### D1: raw vs cleaned data
*Hypothesis:* the raw file teaches the model the corruptions (rupees-for-paise amounts, un-normalised ids, flipped priorities, truncated JSON, v1 schema), so exact match on the 170 dev tickets that are *not* in raw train should drop. The 30 leaked dev tickets can inflate the raw run's official dev number, which is why `exact_clean_dev` is reported too.

In [ ]:
e_d1 = run_or_skip("D1_raw_lora16", raw, desc="RAW train.jsonl (1,445 rows, uncleaned), otherwise as R0")

RUN_EXPERIMENTS=False: D1_raw_lora16 not re-run, its result is in the saved log table


### D2: 16-bit LoRA vs 4-bit QLoRA
*Hypothesis:* NF4 quantisation of the frozen base saves roughly 1.5-2 GB of GPU memory but costs speed (dequantisation) and perhaps a little accuracy. Everything else equals R0.

In [ ]:
e_d2 = run_or_skip("D2_clean_qlora4", clean, quant4=True, desc="cleaned data, 4-bit QLoRA (NF4, double quant, fp16 compute), otherwise as R0")

RUN_EXPERIMENTS=False: D2_clean_qlora4 not re-run, its result is in the saved log table


### D3: number of epochs
*Hypothesis:* the training loss is already ~0.03 after one epoch (R0 log above), so one epoch may be enough and 5 epochs may over-fit the templates and hurt paraphrase robustness. Everything else equals R0.

In [ ]:
e_d3a = run_or_skip("D3a_clean_lora16_1epoch", clean, epochs=1, desc="cleaned data, 16-bit LoRA, 1 epoch (R0: 3)")
e_d3b = run_or_skip("D3b_clean_lora16_5epochs", clean, epochs=5, desc="cleaned data, 16-bit LoRA, 5 epochs (R0: 3)")

RUN_EXPERIMENTS=False: D3a_clean_lora16_1epoch not re-run, its result is in the saved log table
RUN_EXPERIMENTS=False: D3b_clean_lora16_5epochs not re-run, its result is in the saved log table


### D4 (my own idea): emit the derived fields last
The scorer parses the JSON, so key order does not matter for scoring, but it matters for the model: a language model decides each token from the tokens before it. In the spec's order, `priority` and `needs_human` are generated **before** `amount_paise`, so the model must evaluate "amount >= Rs 5,000 / 50,000" implicitly. `priority` is the weakest field in R0 (92%). If the derived fields come *last*, `priority` can attend to the category and the amount digits already written, and `needs_human` can attend to `priority`.

*Change:* only the key order of the training targets: `category, amount_paise, txn_date, txn_id, channel, language, priority, needs_human`. Same 8 keys, same values, same data and hyperparameters. The output is still only the JSON object and no post-processing is involved.

In [ ]:
ORDER = ["category", "amount_paise", "txn_date", "txn_id", "channel", "language", "priority", "needs_human"]
assert sorted(ORDER) == sorted(nl.KEYS)
def reorder(rows, order=ORDER):
    out = []
    for r in rows:
        lab = json.loads(r["messages"][2]["content"])
        out.append({"id": r["id"], "messages": [r["messages"][0], r["messages"][1], {"role": "assistant", "content": json.dumps({k: lab[k] for k in order})}]})
    return out
clean_reordered = reorder(clean)
print(clean[3]["messages"][2]["content"]); print(clean_reordered[3]["messages"][2]["content"])
e_d4 = run_or_skip("D4_clean_lora16_keyorder", clean_reordered, desc="cleaned data, derived fields (priority, needs_human) emitted last, otherwise as R0")

{"category": "refund", "priority": "P1", "amount_paise": 7892400, "txn_date": "2026-08-08", "txn_id": "NP5367592160", "channel": null, "language": "en", "needs_human": true}
{"category": "refund", "amount_paise": 7892400, "txn_date": "2026-08-08", "txn_id": "NP5367592160", "channel": null, "language": "en", "priority": "P1", "needs_human": true}
RUN_EXPERIMENTS=False: D4_clean_lora16_keyorder not re-run, its result is in the saved log table


### Seed noise
Dev has only 200 tickets, so one ticket is 0.5 points and differences of ~2 points are within noise. To calibrate, R0 is repeated with a different seed (different shuffling and LoRA initialisation).

In [ ]:
e_seed = run_or_skip("R0b_clean_lora16_seed1", clean, seed=1, desc="R0 repeated with seed 1 (seed noise)")
saved_table()

RUN_EXPERIMENTS=False: R0b_clean_lora16_seed1 not re-run, its result is in the saved log table


,name,desc,n_train,trainable_M,exact,field_acc,json_valid,exact_clean_dev,exact_rep1.0,peak_gb,train_min
0,B1_base_fixed_prompt,"base model, fixed prompt, no fine-tuning",0,0.000,0.00,0.000,0.00,0.000,NaN,5.954,0.000
1,B2_base_rules_in_prompt,"base model, SCHEMA.md rules in system prompt (...",0,0.000,0.00,0.194,0.34,0.000,NaN,5.954,0.000
2,R0_clean_lora16,"cleaned data, 16-bit LoRA r=16, lr 2e-4, 3 epochs",1320,18.465,0.89,0.979,1.00,0.888,0.905,8.209,14.298
3,D1_raw_lora16,"RAW train.jsonl (1,445 rows, uncleaned), other...",1445,18.465,0.79,0.965,1.00,0.776,0.870,5.308,15.438
4,D2_clean_qlora4,"cleaned data, 4-bit QLoRA (NF4, double quant, ...",1320,18.465,0.88,0.977,1.00,0.882,0.905,4.700,17.666
5,D3a_clean_lora16_1epoch,"cleaned data, 16-bit LoRA, 1 epoch (R0: 3)",1320,18.465,0.42,0.901,1.00,0.424,0.480,5.265,4.774
6,D3b_clean_lora16_5epochs,"cleaned data, 16-bit LoRA, 5 epochs (R0: 3)",1320,18.465,0.93,0.986,1.00,0.924,0.930,5.379,23.783
7,D4_clean_lora16_keyorder,"cleaned data, derived fields (priority, needs_...",1320,18.465,0.87,0.975,1.00,0.876,0.890,5.335,14.292


### Part D: findings
All runs: Colab T4, seed 0, the same 200 dev tickets (one ticket = 0.5 points; the 95% interval of a binomial proportion near 90% on 200 tickets is about +-4 points, so single-seed differences of 1-2 points are not significant).

| Experiment | Result (exact match) | What it shows |
|---|---|---|
| **D1** raw vs cleaned data | 79.0% vs 89.0% (clean-dev 77.6% vs 88.8%) | Cleaning is worth about **+10 points**: this effect is far larger than the noise. The raw-trained model is also fragile: with `repetition_penalty=1.0` it jumps to 87.0% (+8 points), the clean model gains only +1.5. |
| **D2** 16-bit LoRA vs 4-bit QLoRA | 89.0% vs 88.0% | Within noise (2 tickets). QLoRA peaked at 4.7 GB vs 5.3 GB (-0.6 GB) and trained 24% slower (17.7 vs 14.3 min). The memory saving is small, probably because the peak is dominated by activations and the fp32 upcast of the non-quantised layers, not by the weights (not measured separately). |
| **D3** epochs | 1 epoch 42.0%, 3 epochs 89.0%, 5 epochs **93.0%** (clean-dev 92.4%) | My hypothesis (one epoch is enough, more epochs over-fit) was **wrong**. Training loss after 1 epoch (0.024) looks small, but exact match needs every digit right: the per-example loss must be near zero, and it only gets there after 3-5 epochs (final loss 0.0240 / 0.0013 / 0.0002). 5 epochs also removes the repetition-penalty sensitivity (93.0% with and without it). |
| **D4** derived fields last | 87.0% vs 89.0% | **Negative result.** Emitting `priority` and `needs_human` after the amount did not help (-2 points, 4 tickets, within noise). So the priority errors are not caused by the model having to decide before it has written the amount; see the error analysis. |

The final model uses **5 epochs** by a rule fixed before D3b finished (5 epochs only if its clean-dev exact match beats R0's by at least 2 points). Seed-to-seed noise was measured separately on other hardware (see the report; non-T4).

## Part E: error analysis of the best model on dev

In [ ]:
FINAL_EPOCHS = 5            # decided by a rule fixed before seeing D3b: 5 epochs only if D3b's exact match on the 170 clean dev tickets beats R0's by >= 2 points (92.4% vs 88.8% -> yes)
if FINAL_EPOCHS == 3:
    BEST = "R0_clean_lora16"          # the live R0 run above is the final model
else:
    m, e_final = run_experiment(f"FINAL_clean_lora16_{FINAL_EPOCHS}ep", clean, epochs=FINAL_EPOCHS, desc=f"final model: cleaned data, 16-bit LoRA, {FINAL_EPOCHS} epochs")
    del m; nt.free_memory()
    BEST = f"FINAL_clean_lora16_{FINAL_EPOCHS}ep"
print("final model:", BEST)

def dev_errors(tag):
    preds = {p["id"]: p["output"] for p in map(json.loads, open(f"{WORK}/preds/{tag}_dev.jsonl", encoding="utf-8"))}
    errs = []
    for g in dev:
        gold = json.loads(g["messages"][2]["content"])
        try:
            pr = json.loads(preds[g["id"]])
        except Exception:
            pr = None
        bad = [k for k in nl.KEYS if pr is None or k not in pr or pr[k] != gold[k] or type(pr[k]) is not type(gold[k])]
        if bad:
            errs.append({"id": g["id"], "bad": bad, "gold": gold, "pred": pr, "text": g["messages"][1]["content"].split("\n\n", 1)[1], "raw": preds[g["id"]]})
    return errs

errs = dev_errors(BEST)
print(f"{BEST}: {len(errs)} of {len(dev)} dev tickets have at least one wrong field")
print("wrong fields:", collections.Counter(k for e in errs for k in e["bad"]).most_common())
print("tickets by number of wrong fields:", dict(collections.Counter(len(e["bad"]) for e in errs)))
pri = collections.Counter((e["gold"]["priority"], (e["pred"] or {}).get("priority")) for e in errs if "priority" in e["bad"])
print("priority errors (gold -> predicted):", dict(pri))
cat = collections.Counter((e["gold"]["category"], (e["pred"] or {}).get("category")) for e in errs if "category" in e["bad"])
print("category errors (gold -> predicted):", dict(cat))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[FINAL_clean_lora16_5ep] final model: cleaned data, 16-bit LoRA, 5 epochs
  rows 1320 | trainable 18.46M of 1562M | quant4=False r=16 lr=0.0002 epochs=5
  ep 1/5 step   10/415 loss 1.6305 lr 1.00e-04 |g| 1.88  0.6 min
  ep 1/5 step   20/415 loss 0.3115 lr 2.00e-04 |g| 0.63  1.1 min
  ep 1/5 step   30/415 loss 0.0641 lr 2.00e-04 |g| 0.34  1.7 min
  ep 1/5 step   40/415 loss 0.0472 lr 1.99e-04 |g| 0.29  2.4 min
  ep 1/5 step   50/415 loss 0.0356 lr 1.97e-04 |g| 0.24  3.0 min
  ep 1/5 step   60/415 loss 0.0306 lr 1.95e-04 |g| 0.25  3.5 min
  ep 1/5 step   70/415 loss 0.0300 lr 1.92e-04 |g| 0.27  4.1 min
  ep 1/5 step   80/415 loss 0.0221 lr 1.89e-04 |g| 0.12  4.7 min
  ep 2/5 step   90/415 loss 0.0162 lr 1.85e-04 |g| 0.11  5.3 min
  ep 2/5 step  100/415 loss 0.0163 lr 1.80e-04 |g| 0.37  5.8 min
  ep 2/5 step  110/415 loss 0.0117 lr 1.75e-04 |g| 0.11  6.6 min
  ep 2/5 step  120/415 loss 0.0111 lr 1.70e-04 |g| 0.13  7.1 min
  ep 2/5 step  130/415 loss 0.0083 lr 1.64e-04 |g| 0.11  7.7 min
  

In [ ]:
def show(e, width=420):
    print(f"--- {e['id']}  wrong: {e['bad']}")
    print("   ticket:", e["text"].replace(chr(10), " / ")[:width])
    for k in e["bad"]:
        print(f"   {k:13} gold={e['gold'][k]!r:>16}   pred={(e['pred'] or {}).get(k, '<missing>')!r}")
for e in errs[:14]:
    show(e)

def paired(tag_a, tag_b):
    A = {e["id"] for e in dev_errors(tag_a)}; B = {e["id"] for e in dev_errors(tag_b)}
    print(f"{tag_a}: {len(A)} tickets wrong | {tag_b}: {len(B)} wrong | only {tag_a} wrong: {len(A - B)} | only {tag_b} wrong: {len(B - A)} | both wrong: {len(A & B)} | both right: {len(dev) - len(A | B)}")
if BEST != "R0_clean_lora16":
    paired("R0_clean_lora16", BEST)      # paired view of the same 200 tickets: how many flip each way

--- dv-00017  wrong: ['amount_paise']
   ticket: @NimbusPayCare I am still waiting for a refund. Mode: internet banking. My account balance is 712 rupees right now. It took place on 13th June 2026. You can check np5682637619. The amount was Rs 1,80,827.25.
   amount_paise  gold=        18082725   pred=18082525
--- dv-00019  wrong: ['category']
   ticket: Dear NimbusPay support, My product was faulty. Amount: 259419 rupees I am on the iOS app. Mode: credit card. Kindly resolve this asap.
   category      gold=        'refund'   pred='product_fault'
--- dv-00037  wrong: ['priority', 'needs_human']
   ticket: My transaction history is looking a little off. I have been charged twice for multiple different purchases, and would like to have those duplicate charges refunded. The txn id is np 42846 07517. Transaction date 6 Sep. 284669 rupees was the value. Waiting for your reply.
   priority      gold=            'P1'   pred='P2'
   needs_human   gold=            True   pred=False
--- dv-0004

## Final model: predictions for dev and test
The final adapter is reloaded from disk exactly as the graders will do it (base model in fp16 + PEFT adapter), then decoded with plain greedy `generate`, `max_new_tokens=200`, one ticket at a time (no padding), the model's own chat template and generation_config, and only `.strip()` applied to the text.

In [ ]:
from peft import PeftModel
nt.free_memory(); torch.cuda.reset_peak_memory_stats()
base = nt.load_base(quant4=False)
final = PeftModel.from_pretrained(base, f"{WORK}/adapters/{BEST}")
trainable_final = sum(p.numel() for p in final.parameters() if p.requires_grad)
print("adapter loaded from", f"{WORK}/adapters/{BEST}", "| files:", sorted(os.listdir(f"{WORK}/adapters/{BEST}")))
print("adapter parameters:", sum(p.numel() for n, p in final.named_parameters() if "lora_" in n) / 1e6, "M")
t0 = time.time(); dev_out1 = nt.generate(final, tok, dev, batch_size=1); print(f"dev, batch size 1: {(time.time() - t0) / 60:.1f} min")
ev = evaluate(dev, dev_out1, f"{BEST}_final_dev_bs1")
print(f"dev (batch size 1): exact {ev['exact']:.1%} | field acc {ev['field_acc']:.1%} | valid {ev['json_valid']:.1%}")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

adapter loaded from /content/drive/MyDrive/nimbus/work/adapters/FINAL_clean_lora16_5ep | files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors']
adapter parameters: 18.464768 M
dev, batch size 1: 16.7 min
dev (batch size 1): exact 94.0% | field acc 98.9% | valid 100.0%


In [ ]:
t0 = time.time(); test_out = nt.generate(final, tok, test, batch_size=1); print(f"test, batch size 1: {(time.time() - t0) / 60:.1f} min")
with open(f"{WORK}/predictions_test.jsonl", "w", encoding="utf-8") as f:
    for r, o in zip(test, test_out):
        f.write(json.dumps({"id": r["id"], "output": o}, ensure_ascii=False) + chr(10))
valid = sum(nl.parse_label(o) is not None and set(nl.parse_label(o)) == set(nl.KEYS) for o in test_out)
print("test predictions written:", len(test_out), "| valid 8-key JSON objects:", valid)
print(test_out[0])

test, batch size 1: 33.4 min
test predictions written: 400 | valid 8-key JSON objects: 400
{"category": "fraud", "priority": "P1", "amount_paise": null, "txn_date": "2026-07-00", "txn_id": "NP6090913363", "channel": null, "language": "hi-en", "needs_human": true}


**Robustness checks of the submitted predictions.** (1) Does batching change the greedy output? (2) How much does the model's default `repetition_penalty=1.1` change test outputs compared with 1.0? The submitted file uses the default generation_config, i.e. what plain `model.generate(..., do_sample=False)` does.

In [ ]:
t0 = time.time()
test_b16 = nt.generate(final, tok, test, batch_size=16)
test_rp1 = nt.generate(final, tok, test, batch_size=16, repetition_penalty=1.0)
print(f"({(time.time() - t0) / 60:.1f} min)")
print("identical to the submitted file: batch size 16 vs 1:", sum(a == b for a, b in zip(test_out, test_b16)), "/", len(test))
print("identical to the submitted file: repetition_penalty 1.0 vs default 1.1:", sum(a == b for a, b in zip(test_out, test_rp1)), "/", len(test))

(6.1 min)
identical to the submitted file: batch size 16 vs 1: 400 / 400
identical to the submitted file: repetition_penalty 1.0 vs default 1.1: 388 / 400
